# 02 - Feature Extraction

This notebook extracts Concerto Small features for one selected S3DIS area. It is designed to run on a Google Colab T4 instance.

Ensure you have access to the shared Google Drive with the preprocessed data.

### 1. Setup & Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'dev/eval-demo'
AREA_ID = 4
AREA_NAME = f'Area_{AREA_ID}'
FEATURES_NAME = f's3dis_area{AREA_ID}'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


Mounted at /content/drive
Cloning into '/content/Deep_learning_project'...
remote: Enumerating objects: 233, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 233 (delta 135), reused 146 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (233/233), 30.81 MiB | 25.61 MiB/s, done.
Resolving deltas: 100% (135/135), done.


In [2]:
%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

/content/Deep_learning_project
Branch 'dev/enc-mlp' set up to track remote branch 'dev/enc-mlp' from 'origin'.
Switched to a new branch 'dev/enc-mlp'
From https://github.com/Gandata/Deep_learning_project
 * branch            dev/enc-mlp -> FETCH_HEAD
Already up to date.
dev/enc-mlp
06928ef (HEAD -> dev/enc-mlp, origin/dev/enc-mlp) fix(extraction): chunk large S3DIS rooms to avoid Colab OOM


In [3]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto

Cloning into '/content/Concerto'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 74 (delta 10), reused 23 (delta 9), pack-reused 42 (from 1)
Receiving objects: 100% (74/74), 2.12 MiB | 8.42 MiB/s, done.
Resolving deltas: 100% (14/14), done.


### 2. Symlink Data & Checkpoints
We map the Drive folders directly into the repository so our scripts can find them.

In [4]:
# Symlink Drive data
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features

In [5]:
%cd notebooks/

/content/Deep_learning_project/notebooks


### 3. Setup Hugging Face Token
Input your Hugging Face token below to allow `open_clip_torch` to download the models.

In [6]:
# If token is in colab secrets

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

### 4. Run Feature Extraction
This script will iterate over the selected S3DIS area and extract the features. It is resume-safe, so if Colab disconnects, you can just run it again.

In [7]:
# Forzar a uv a usar/instalar Python 3.10.12 y sincronizar
!uv python install 3.10.12
!uv sync --python 3.10.12

Installed Python 3.10.12 in 1.74s
 + cpython-3.10.12-linux-x86_64-gnu (python3.10)
Using CPython 3.10.12
Creating virtual environment at: .venv
Resolved 141 packages in 11.85s
Prepared 134 packages in 1m 38s
Installed 135 packages in 1.09s
 + addict==2.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.13.0
 + attrs==26.1.0
 + blinker==1.9.0
 + brotli==1.2.0
 + camtools==0.1.8
 + ccimport==0.4.4
 + certifi==2026.4.22
 + charset-normalizer==3.4.7
 + click==8.3.3
 + configargparse==1.7.5
 + contourpy==1.3.2
 + cumm-cu120==0.4.11
 + cycler==0.12.1
 + dash==4.1.0
 + exceptiongroup==1.3.1
 + fast-pytorch-kmeans==0.2.2
 + fastapi==0.136.1
 + fastjsonschema==2.21.2
 + filelock==3.29.0
 + fire==0.7.1
 + flask==3.1.3
 + fonttools==4.62.1
 + fsspec==2026.4.0
 + gradio==6.14.0
 + gradio-client==2.5.0
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.4.1
 + hf-xet==1.4.3
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.13.0
 + idna==3.13
 + imageio==2.37.3
 + importlib-metada

In [8]:
!uv add dotenv

Resolved 143 packages in 617ms
Prepared 2 packages in 90ms
Installed 2 packages in 3ms
 + dotenv==0.9.9
 + python-dotenv==1.2.2


In [9]:
# Run the real extraction script with chunking to avoid T4 OOM on large rooms
print('Selected area:', AREA_NAME)
print('Output feature folder:', FEATURES_NAME)
!PYTHONPATH=/content/Concerto uv run python /content/Deep_learning_project/scripts/extract_features.py --data_dir /content/Deep_learning_project/data/s3dis_processed --out_dir /content/Deep_learning_project/features/{FEATURES_NAME} --areas {AREA_ID} --feature-dtype float16 --max-points-per-chunk 100000

Loading Area 5 from /content/Deep_learning_project/data/s3dis_processed...
S3DISDataset: 68 rooms loaded from areas [5]
/content/Deep_learning_project/notebooks/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/content/Deep_learning_project/notebooks/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/content/Deep_learning_project/notebooks/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/co

auto: notebook execution

# Other tests

In [10]:
#!PYTHONPATH=/content/Concerto uv run python /content/Concerto/demo/0_pca.py

In [11]:
#!PYTHONPATH=/content/Concerto uv run python -c "import sys, torch; sys.path.insert(0, '/content/Concerto'); import concerto; import spconv.pytorch as spconv; print('python ok'); print(torch.__version__, torch.version.cuda); print(spconv.__file__)"